# SO₂ Data Cleaning and Validation (2020–2023)

This notebook performs cleaning and validation of hourly SO₂ data collected from 2020 to 2023.

The objective is to:
- Combine yearly datasets
- Clean invalid and missing values
- Convert hourly data to numeric format
- Calculate daily average SO₂ levels
- Remove duplicates and missing records
- Create time-based features (Year, Month, Season)
- Export a validated city-day dataset

The final output will be used for merging with NO₂ and PM2.5 datasets for further analysis.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../../data/raw")

files = sorted(RAW_DIR.glob("SO2_*.csv"))

print("SO2 files found:", [f.name for f in files])

df_list = []

for file in files:
    temp = pd.read_csv(file, skiprows=7)
    temp.columns = temp.columns.str.strip()
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Rows:", df.shape[0], "Cols:", df.shape[1])
df.head()


SO2 files found: ['SO2_2020.csv', 'SO2_2021.csv', 'SO2_2022.csv', 'SO2_2023.csv']
Rows: 210018 Cols: 31


,Pollutant//Polluant,NAPS ID//Identifiant SNPA,City//Ville,Province/Territory//Province/Territoire,Latitude//Latitude,Longitude//Longitude,Date//Date,H01//H01,H02//H02,H03//H03,...,H15//H15,H16//H16,H17//H17,H18//H18,H19//H19,H20//H20,H21//H21,H22//H22,H23//H23,H24//H24
0,SO2,10102,St. John's,NL,47.56038,-52.71147,2020-01-01,0.1,0.1,0.1,...,0.1,0.1,0.0,0.0,0.1,0.1,0.1,0.1,0.1,0.2
1,SO2,10102,St. John's,NL,47.56038,-52.71147,2020-01-02,0.1,0.1,0.1,...,2.0,1.1,0.7,0.9,0.5,0.4,0.3,0.3,0.3,0.3
2,SO2,10102,St. John's,NL,47.56038,-52.71147,2020-01-03,0.4,0.5,1.0,...,0.3,0.3,0.3,0.2,0.3,0.2,0.2,0.2,0.2,0.2
3,SO2,10102,St. John's,NL,47.56038,-52.71147,2020-01-04,0.2,0.3,0.3,...,0.7,0.4,0.4,0.3,0.3,0.3,0.3,0.2,0.3,0.5
4,SO2,10102,St. John's,NL,47.56038,-52.71147,2020-01-05,0.5,0.6,0.8,...,0.4,0.2,0.2,0.2,0.2,0.2,0.2,0.3,0.2,0.3


In [2]:
df.replace(-999, np.nan, inplace=True)

In [3]:
hour_cols = [c for c in df.columns if "H" in c and "//" in c]
hour_cols = sorted(hour_cols)

print("Hourly columns:", len(hour_cols))
hour_cols[:5]


Hourly columns: 24


['H01//H01', 'H02//H02', 'H03//H03', 'H04//H04', 'H05//H05']

In [4]:
for c in hour_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [5]:
df["SO2_daily_avg"] = df[hour_cols].mean(axis=1)
df[["SO2_daily_avg"]].describe()

,SO2_daily_avg
count,203043.000000
mean,0.716169
std,1.946725
min,-0.079167
25%,0.108333
50%,0.254167
75%,0.565217
max,67.600000


In [6]:
df.rename(columns={
    "Date//Date": "Date",
    "City//Ville": "City"
}, inplace=True)

final = df[["Date", "City", "SO2_daily_avg"]].copy()

print("Final shape:", final.shape)
final.head()

Final shape: (210018, 3)


,Date,City,SO2_daily_avg
0,2020-01-01,St. John's,0.066667
1,2020-01-02,St. John's,0.458333
2,2020-01-03,St. John's,0.470833
3,2020-01-04,St. John's,0.766667
4,2020-01-05,St. John's,0.666667


In [7]:
final["Date"] = pd.to_datetime(final["Date"], errors="coerce")

final["Year"] = final["Date"].dt.year
final["Month"] = final["Date"].dt.month

def season_from_month(m):
    if m in [12,1,2]:
        return "Winter"
    elif m in [3,4,5]:
        return "Spring"
    elif m in [6,7,8]:
        return "Summer"
    else:
        return "Fall"

final["Season"] = final["Month"].apply(season_from_month)

print("Final columns:", final.columns)
final.head()

Final columns: Index(['Date', 'City', 'SO2_daily_avg', 'Year', 'Month', 'Season'], dtype='object')


,Date,City,SO2_daily_avg,Year,Month,Season
0,2020-01-01,St. John's,0.066667,2020,1,Winter
1,2020-01-02,St. John's,0.458333,2020,1,Winter
2,2020-01-03,St. John's,0.470833,2020,1,Winter
3,2020-01-04,St. John's,0.766667,2020,1,Winter
4,2020-01-05,St. John's,0.666667,2020,1,Winter


In [8]:
OUTPUT_DIR = Path("../../data/validated")
OUTPUT_DIR.mkdir(exist_ok=True)

final.to_csv(OUTPUT_DIR / "SO2_cityday.csv", index=False)

print("SO2 validated dataset saved successfully!")

SO2 validated dataset saved successfully!
